# Склад синтетичного набору і чутливість детектора дефектів

Серія навчань для статті: 9 складів навчального набору × 3 сіди = 27 моделей
MobileNetV2, оцінка кожної на спільних тестових наборах.

**Як запустити**

1. *Середовище виконання → Змінити тип середовища виконання → GPU (T4).*
2. *Середовище виконання → Виконати все.* Дозвольте доступ до Google Диска:
   результати пишуться в `MyDrive/defect_study`, тож якщо сесія обірветься,
   просто запустіть усе знову - готові моделі пропускаються.
3. Близько години. В кінці завантажиться `defect_study_results.zip` - покладіть
   його вміст у `article/results/` репозиторію.

Для швидкої перевірки, що все працює, поставте `QUICK = True` (кілька хвилин).

In [ ]:
import os

try:
    from google.colab import drive, files
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/defect_study'
except ImportError:
    files = None
    ROOT = os.path.abspath('defect_study')

for sub in ('scores', 'models'):
    os.makedirs(f'{ROOT}/{sub}', exist_ok=True)
print('результати:', ROOT)

Генератор силуетів і модель - копії `experiments/shapes.py` і `experiments/model.py` з репозиторію.

In [ ]:
%%writefile shapes.py
"""Параметричний генератор силуетів: фігури з дефектами розміру h і нерівністю контуру eps."""
import numpy as np
from PIL import Image, ImageDraw

IMG_SIZE = 128
CANVAS = 512
CENTER = CANVAS / 2

SHAPES = ('square', 'rectangle', 'circle', 'ellipse', 'pentagon', 'hexagon')
EXCESS = ('line', 'bump', 'blob')
MISSING = ('slit', 'dent')
DEFECTS = EXCESS + MISSING

TRAIN_RANGES = {
    'line': (0.35, 0.70),
    'bump': (0.12, 0.22),
    'blob': (0.46, 0.77),
    'slit': (0.20, 0.45),
    'dent': (0.12, 0.22),
}


def ellipse_outline(rx, ry, steps=96):
    t = np.linspace(0, 2 * np.pi, steps, endpoint=False)
    return list(zip(rx * np.cos(t), ry * np.sin(t)))


def unit_shape(kind, rng):
    if kind == 'square':
        return [(-1, -1), (1, -1), (1, 1), (-1, 1)]
    if kind == 'rectangle':
        b = rng.uniform(0.45, 0.72)
        return [(-1, -b), (1, -b), (1, b), (-1, b)]
    if kind == 'circle':
        return ellipse_outline(1.0, 1.0)
    if kind == 'ellipse':
        return ellipse_outline(1.0, rng.uniform(0.5, 0.78))
    sides = 5 if kind == 'pentagon' else 6
    angles = -np.pi / 2 + np.arange(sides) * 2 * np.pi / sides
    return list(zip(np.cos(angles), np.sin(angles)))


def place(points, scale, rotation, center=CENTER):
    c, s = np.cos(rotation), np.sin(rotation)
    return np.array([(center + scale * (x * c - y * s), center + scale * (x * s + y * c))
                     for x, y in points])


def densify(points, step):
    out = []
    for p, q in zip(points, np.roll(points, -1, axis=0)):
        k = max(1, int(np.ceil(np.hypot(*(q - p)) / step)))
        out.extend(p + (q - p) * j / k for j in range(k))
    return np.array(out)


def wobble(points, amp, rng, k_min=2, k_max=8):
    d = points - CENTER
    r = np.hypot(d[:, 0], d[:, 1])
    theta = np.arctan2(d[:, 1], d[:, 0])
    ks = np.arange(k_min, k_max + 1)
    a = rng.normal(size=len(ks)) / ks
    phi = rng.uniform(0, 2 * np.pi, len(ks))
    f = (a * np.cos(np.outer(theta, ks) + phi)).sum(axis=1)
    f /= np.abs(f).max()
    return CENTER + d * ((r + amp * f) / r)[:, None]


def anchor(points, smooth, rng, where=None):
    """Точка контуру і зовнішня нормаль у ній; where = (номер ребра, частка вздовж нього)."""
    edges = np.roll(points, -1, axis=0) - points
    lengths = np.hypot(edges[:, 0], edges[:, 1])
    if where is None:
        i = rng.choice(len(points), p=lengths / lengths.sum())
        u = rng.uniform(0, 1) if smooth else rng.uniform(0.2, 0.8)
    else:
        i, u = where
    a = points[i] + u * edges[i]
    n = np.array([edges[i][1], -edges[i][0]]) / lengths[i]
    if np.dot(n, a - points.mean(axis=0)) < 0:
        n = -n
    return a, n


def segment(p0, p1, width):
    d = (p1 - p0) / np.hypot(*(p1 - p0))
    o = np.array([-d[1], d[0]]) * width / 2
    return [tuple(p0 + o), tuple(p1 + o), tuple(p1 - o), tuple(p0 - o)]


def disc(c, r):
    return [c[0] - r, c[1] - r, c[0] + r, c[1] + r]


def render(kind, rng, defect=None, h=None, eps=0.0, info=None, scale=None, rotation=None, where=None):
    image = Image.new('L', (CANVAS, CANVAS), 255)
    pen = ImageDraw.Draw(image)

    s = CANVAS * 0.28 * (rng.uniform(0.85, 1.15) if scale is None else scale)
    nominal = place(unit_shape(kind, rng), s, rng.uniform(0, 2 * np.pi) if rotation is None else rotation)
    outline = wobble(densify(nominal, 4.0), eps * s, rng) if eps > 0 else nominal
    pen.polygon([tuple(p) for p in outline], fill=0)
    if info is not None:
        info.update(s=s, nominal=nominal)

    if defect is None:
        return image

    a, n = anchor(nominal, smooth=kind in ('circle', 'ellipse'), rng=rng, where=where)
    reach = h * s
    if defect == 'line':
        pen.polygon(segment(a - n * s * 0.06, a + n * reach, s * rng.uniform(0.04, 0.09)), fill=0)
    elif defect == 'bump':
        pen.ellipse(disc(a, reach), fill=0)
    elif defect == 'blob':
        r = 0.3 * reach
        c = a + n * (reach - r)
        stalk = min(s * rng.uniform(0.03, 0.07), 1.2 * r)
        pen.polygon(segment(a - n * s * 0.06, c, stalk), fill=0)
        pen.ellipse(disc(c, r), fill=0)
    elif defect == 'slit':
        pen.polygon(segment(a + n * s * 0.06, a - n * reach, s * rng.uniform(0.04, 0.09)), fill=255)
    elif defect == 'dent':
        pen.ellipse(disc(a, reach), fill=255)
    else:
        raise ValueError(f'невідомий дефект: {defect}')
    return image


def fit_to_frame(image):
    pixels = np.array(image)
    ys, xs = np.where(pixels < 128)
    if len(xs) == 0:
        return None
    crop = image.crop((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1))
    w, h = crop.size
    scale = IMG_SIZE / max(w, h)
    crop = crop.resize((max(1, round(w * scale)), max(1, round(h * scale))), Image.BILINEAR)
    frame = Image.new('L', (IMG_SIZE, IMG_SIZE), 255)
    frame.paste(crop, ((IMG_SIZE - crop.size[0]) // 2, (IMG_SIZE - crop.size[1]) // 2))
    return np.where(np.array(frame) < 128, 0, 255).astype(np.uint8)


def sample(kind, rng, defect=None, h=None, eps=0.0):
    return fit_to_frame(render(kind, rng, defect, h, eps))


def log_uniform(rng, lo, hi):
    return float(np.exp(rng.uniform(np.log(lo), np.log(hi))))


def composition_set(types, n, rng, ranges=TRAIN_RANGES):
    images, labels = np.empty((n, IMG_SIZE, IMG_SIZE), np.uint8), np.empty(n, np.float32)
    for i in range(n):
        kind = SHAPES[rng.integers(len(SHAPES))]
        if rng.random() < 0.5:
            images[i], labels[i] = sample(kind, rng), 0.0
        else:
            defect = types[rng.integers(len(types))]
            images[i] = sample(kind, rng, defect, rng.uniform(*ranges[defect]))
            labels[i] = 1.0
    return images, labels


def tolerance_set(T, n, rng):
    """Явний допуск T: відхилення h < T і eps < T - норма, h >= T - брак."""
    small = ('line', 'bump', 'slit', 'dent')
    images, labels = np.empty((n, IMG_SIZE, IMG_SIZE), np.uint8), np.empty(n, np.float32)
    for i in range(n):
        kind = SHAPES[rng.integers(len(SHAPES))]
        part = i % 3
        if part == 0:
            eps = rng.uniform(0, T) if rng.random() < 0.5 else 0.0
            images[i], labels[i] = sample(kind, rng, eps=eps), 0.0
        elif part == 1:
            defect = small[rng.integers(len(small))]
            images[i], labels[i] = sample(kind, rng, defect, log_uniform(rng, 0.01, T)), 0.0
        else:
            defect = DEFECTS[rng.integers(len(DEFECTS))]
            lo = max(T, 0.25) if defect == 'blob' else T
            hi = 0.77 if defect == 'blob' else 0.5
            images[i], labels[i] = sample(kind, rng, defect, log_uniform(rng, lo, hi)), 1.0
    order = rng.permutation(n)
    return images[order], labels[order]


GRID_H = (0.02, 0.03, 0.05, 0.07, 0.10, 0.14, 0.20, 0.28, 0.40)
BLOB_H = (0.20, 0.28, 0.40)
GRID_EPS = (0.01, 0.02, 0.03, 0.05, 0.08)


def test_suite(rng, per_shape_clean=200, per_cell=120, per_eps=600):
    images, meta = [], []

    def add(kind, set_name, defect=None, h=None, eps=0.0):
        images.append(sample(kind, rng, defect, h, eps))
        meta.append({'set': set_name, 'shape': kind, 'defect': defect or '',
                     'h': h if h is not None else 0.0, 'eps': eps,
                     'label': 1 if defect else 0})

    for kind in SHAPES:
        for _ in range(per_shape_clean):
            add(kind, 'clean')
    for defect in DEFECTS:
        for h in (BLOB_H if defect == 'blob' else GRID_H):
            for j in range(per_cell):
                add(SHAPES[j % len(SHAPES)], 'grid', defect, h)
    for eps in GRID_EPS:
        for j in range(per_eps):
            add(SHAPES[j % len(SHAPES)], 'irregular', eps=eps)
    return np.stack(images), meta


def clean_set(n, rng):
    return np.stack([sample(SHAPES[i % len(SHAPES)], rng) for i in range(n)])

In [ ]:
%%writefile model.py
"""MobileNetV2 (ImageNet) + GAP + Dense - детектор дефектів, як у базовій роботі."""
import keras

from shapes import IMG_SIZE


def build_model(weights='imagenet'):
    base = keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=weights)
    base.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), dtype='float32', name='image')
    x = keras.layers.Rescaling(1 / 127.5, offset=-1)(inputs)
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(128, activation='relu')(x)
    x = keras.layers.Dropout(0.2)(x)
    outputs = keras.layers.Dense(1, activation='sigmoid', name='defect')(x)
    return keras.Model(inputs, outputs, name='defect_classifier'), base


def load_trained(path):
    model, _ = build_model(weights=None)
    model.load_weights(path)
    return model

In [ ]:
import json
import time

import keras
import numpy as np
import pandas as pd
import tensorflow as tf

import shapes as S
from model import build_model

print('TensorFlow', tf.__version__, '| Keras', keras.__version__,
      '| GPU:', tf.config.list_physical_devices('GPU'))

QUICK = False

COMPOSITIONS = {
    'L':   {'types': ['line']},
    'B':   {'types': ['bump']},
    'D':   {'types': ['dent']},
    'M':   {'types': ['line', 'blob']},
    'S':   {'types': ['line', 'bump', 'blob']},
    'A':   {'types': list(S.DEFECTS)},
    'T05': {'tolerance': 0.05},
    'T10': {'tolerance': 0.10},
    'T15': {'tolerance': 0.15},
}
SEEDS = [0, 1, 2]
N_SAMPLES = 6000
EPOCHS_HEAD, EPOCHS_TUNE = 20, 8
BATCH = 64
SUITE = dict(per_shape_clean=200, per_cell=120, per_eps=600)
N_CALIB = 1200

if QUICK:
    ROOT += '_quick'
    for sub in ('scores', 'models'):
        os.makedirs(f'{ROOT}/{sub}', exist_ok=True)
    COMPOSITIONS = {k: COMPOSITIONS[k] for k in ('S', 'T10')}
    SEEDS, N_SAMPLES, EPOCHS_HEAD, EPOCHS_TUNE = [0], 120, 1, 1
    SUITE, N_CALIB = dict(per_shape_clean=4, per_cell=6, per_eps=6), 24

with open(f'{ROOT}/config.json', 'w') as fh:
    json.dump({'compositions': COMPOSITIONS, 'seeds': SEEDS, 'n_samples': N_SAMPLES,
               'epochs': [EPOCHS_HEAD, EPOCHS_TUNE], 'train_ranges': S.TRAIN_RANGES,
               'grid_h': S.GRID_H, 'blob_h': S.BLOB_H, 'grid_eps': S.GRID_EPS,
               'suite': SUITE, 'n_calib': N_CALIB}, fh, indent=1)

## Спільні тестові набори

Генеруються один раз з окремим сідом і кешуються на Диску:
ідеальні фігури, сітка «тип дефекту × розмір h», нерівні фігури (eps)
та окремий набір ідеальних фігур для калібрування порога на FPR = 1 %.

In [ ]:
suite_path = f'{ROOT}/test_suite.npz'
if os.path.exists(suite_path):
    cached = np.load(suite_path)
    X_test, X_calib = cached['X_test'], cached['X_calib']
    meta = pd.read_csv(f'{ROOT}/test_meta.csv', index_col='id', keep_default_na=False)
else:
    X_test, meta = S.test_suite(np.random.default_rng(2026), **SUITE)
    X_calib = S.clean_set(N_CALIB, np.random.default_rng(2027))
    meta = pd.DataFrame(meta)
    meta.to_csv(f'{ROOT}/test_meta.csv', index_label='id')
    np.savez_compressed(suite_path, X_test=X_test, X_calib=X_calib)

print('тест:', X_test.shape, '| калібрування:', X_calib.shape)
print(meta.groupby(['set', 'defect']).size())

## Модель і навчання - як у базовій роботі, рання зупинка за val loss

In [ ]:
@tf.autograph.experimental.do_not_convert
def with_rgb(image, label):
    return tf.image.grayscale_to_rgb(tf.cast(image, tf.float32)), label


@tf.autograph.experimental.do_not_convert
def only_rgb(image):
    return tf.image.grayscale_to_rgb(tf.cast(image, tf.float32))


def to_dataset(images, labels=None, shuffle=False, seed=0):
    x = images[..., None]
    if labels is None:
        data = tf.data.Dataset.from_tensor_slices(x).batch(256).map(only_rgb)
    else:
        data = tf.data.Dataset.from_tensor_slices((x, labels))
        if shuffle:
            data = data.shuffle(4096, seed=seed)
        data = data.batch(BATCH).map(with_rgb)
    return data.prefetch(tf.data.AUTOTUNE)


def fit(model, base, train_ds, val_ds):
    def stop():
        return keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=4, restore_best_weights=True)

    def compile_with(rate):
        model.compile(optimizer=keras.optimizers.Adam(rate), loss='binary_crossentropy',
                      metrics=['accuracy', keras.metrics.AUC(name='auc')])

    # перемішує tf.data; shuffle=False лише прибирає попередження
    compile_with(1e-3)
    head = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                     callbacks=[stop()], shuffle=False, verbose=0)

    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    compile_with(1e-5)
    tune = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_TUNE,
                     callbacks=[stop()], shuffle=False, verbose=0)
    return head, tune


def predict(model, images):
    return model.predict(to_dataset(images), verbose=0).ravel().astype(np.float32)


def auc_of(labels, scores):
    metric = keras.metrics.AUC()
    metric.update_state(labels, scores)
    return float(metric.result())

## Серія навчань

Для кожного складу і сіду: генерація 6000 зразків (60/20/20), навчання,
передбачення на власному тесті, на спільних тестових наборах і на
калібрувальному наборі. Усе зберігається одразу, тож перезапуск продовжує
з місця обриву.

In [ ]:
log_path = f'{ROOT}/train_log.csv'
log = pd.read_csv(log_path).to_dict('records') if os.path.exists(log_path) else []
done = {(row['comp'], row['seed']) for row in log}

for ci, (comp, spec) in enumerate(COMPOSITIONS.items()):
    for seed in SEEDS:
        name = f'{comp}_s{seed}'
        if (comp, seed) in done:
            print('вже є:', name)
            continue

        started = time.time()
        keras.utils.set_random_seed(1000 * seed + ci)
        rng = np.random.default_rng(10_000 + 100 * ci + seed)
        if 'tolerance' in spec:
            X, y = S.tolerance_set(spec['tolerance'], N_SAMPLES, rng)
        else:
            X, y = S.composition_set(spec['types'], N_SAMPLES, rng)

        n_test = n_val = N_SAMPLES // 5
        X_own, y_own = X[:n_test], y[:n_test]
        X_val, y_val = X[n_test:n_test + n_val], y[n_test:n_test + n_val]
        X_train, y_train = X[n_test + n_val:], y[n_test + n_val:]

        model, base = build_model()
        head, tune = fit(model, base,
                         to_dataset(X_train, y_train, shuffle=True, seed=seed),
                         to_dataset(X_val, y_val))

        p_own = predict(model, X_own)
        np.savez_compressed(f'{ROOT}/scores/{name}.npz',
                            test=predict(model, X_test), calib=predict(model, X_calib),
                            own=p_own, own_labels=y_own)
        # копія без стану оптимізатора: файл удвічі менший
        bare, _ = build_model(weights=None)
        bare.set_weights(model.get_weights())
        bare.save_weights(f'{ROOT}/models/{name}.weights.h5')

        row = {'comp': comp, 'seed': seed,
               'epochs_head': len(head.history['loss']),
               'epochs_tune': len(tune.history['loss']),
               'val_loss': round(float(min(tune.history['val_loss'])), 5),
               'own_acc': round(float(((p_own > 0.5) == y_own).mean()), 4),
               'own_auc': round(auc_of(y_own, p_own), 4),
               'seconds': round(time.time() - started, 1)}
        log.append(row)
        pd.DataFrame(log).to_csv(log_path, index=False)
        print(row, flush=True)

        del model, base, bare
        keras.backend.clear_session()

## Короткий підсумок

Сід 0, поріг 0.5: частка виявлених дефектів кожного типу при h ≥ 0.10
і частка хибних тривог на ідеальних фігурах. Повний аналіз - локально.

In [ ]:
grid = meta['set'] == 'grid'
big = grid & (meta['h'] >= 0.10)
rows = []
for comp in COMPOSITIONS:
    scores = np.load(f'{ROOT}/scores/{comp}_s0.npz')['test']
    row = {'склад': comp, 'FPR ідеальні': (scores[meta['set'] == 'clean'] > 0.5).mean()}
    for defect in S.DEFECTS:
        row[defect] = (scores[big & (meta['defect'] == defect)] > 0.5).mean()
    rows.append(row)
print(pd.DataFrame(rows).round(3).to_string(index=False))
print()
print(pd.read_csv(log_path).groupby('comp')[['own_acc', 'own_auc', 'seconds']].mean().round(4))

## Пакування результатів

В архів ідуть оцінки всіх моделей, метадані тестів, журнал навчання і моделі
з сідом 0 (для реальних фото). Усе інше лишається на Диску в `defect_study`.

In [ ]:
import shutil

pack = '/content/defect_study_results' if files else os.path.abspath('defect_study_results')
shutil.rmtree(pack, ignore_errors=True)
os.makedirs(f'{pack}/models')
for name in ('config.json', 'test_meta.csv', 'train_log.csv'):
    shutil.copy(f'{ROOT}/{name}', pack)
shutil.copytree(f'{ROOT}/scores', f'{pack}/scores')
for comp in COMPOSITIONS:
    shutil.copy(f'{ROOT}/models/{comp}_s0.weights.h5', f'{pack}/models')

archive = shutil.make_archive(pack, 'zip', pack)
print(archive, round(os.path.getsize(archive) / 2**20, 1), 'МБ')
if files:
    files.download(archive)